In [3]:
%matplotlib inline
import jax.numpy as jnp
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, "../")  
from TOOL_BOX_OF_FUNCTIONS.GAUSSIAN_POSTERIOR_FUNCTION import POSTERIOR as POST 
from TOOL_BOX_OF_FUNCTIONS.BAYESIAN_REGRESSION import BAYES_REG as BR 

Part 1 — Bayesian linear regression with mixed basis

We observe 
$$
y_n = f(x_n) + \varepsilon_n, \quad \varepsilon_n \sim \mathcal{N}(0, \sigma^2)
$$
and
$$
f(x) = w_0 + w_1 x + w_2 x^2 + w_3 \sin(0.5x) 
= w^\top \phi(x), 
\quad \phi(x) = [1, \, x, \, x^2, \, \sin(0.5x)]^\top.
$$

Data ($N = 6$):
$$
x = [-2.2, \, -0.7, \, 0.0, \, 1.3, \, 2.4, \, 3.0]
$$
$$
y = [-0.9, \, -0.1, \, 0.2, \, 1.5, \, 2.6, \, 2.7]
$$

Priors
$$
w_j \stackrel{\text{iid}}{\sim} \mathcal{N}(0, \alpha^{-1}), \quad \alpha = 1.
$$
Noise: $\sigma = 0.30$ (fixed).  
Let $\Phi \in \mathbb{R}^{N \times 4}$ be the design matrix.

---

Q1.1 (Easy)
Write the analytical likelihood $p(y \mid w, \Phi, \sigma^2)$ and the marginal likelihood $p(y \mid \alpha, \sigma)$ in closed form.

Hint: Linear–Gaussian identities. Your expressions should make the roles of $\Phi$, $\alpha$, and $\sigma$ explicit.  
(Study: Lecture 4 ``Bayesian linear regression: key equations'' \& ``Marginal likelihood'').

---

Q1.2 (Medium)
Compute the MLE $\hat{w}_{\mathrm{ML}}$. Report the fitted mean 
$$
\hat{f}(x) = \hat{w}_{\mathrm{ML}}^\top \phi(x)
$$
at $x = 1.0$.

Hint: Solve $(\Phi^\top \Phi) \hat{w} = \Phi^\top y$.  
(Study: derivation vs. plug-in predictive on slides 5–6).

---

Q1.3 (Medium)
Derive the posterior $p(w \mid y)$ and report the \textbf{posterior mean} $m$ and the \textbf{marginal posterior standard deviations} for each $w_j$.

Hint:
$$
S = (\alpha I + \beta \Phi^\top \Phi)^{-1}, 
\quad m = \beta S \Phi^\top y, 
\quad \beta = \sigma^{-2}.
$$
(Study: slide 4).

---

Q1.4 (Medium)
Compute the posterior predictive $p(y_* \mid y, x_*)$ at $x_* = 2.0$.  
Report the \textbf{mean}, \textbf{SD}, and a \textbf{95\% credibility interval}.

Hint:
$$
\mathrm{Var}(y_* \mid y) = \phi_*^\top S \phi_* + \sigma^2.
$$
(Study: slides 5–6).

---

Q1.5 (Hard, reformulated)
Suppose a colleague insists on using the \textit{plug-in MAP} instead of the full posterior predictive.
Give conditions under which the two predictive means and variances will be close, and justify using equations.

Hint: Consider the size of $S$ and effective data size/conditioning of $\Phi^\top \Phi$.
(Study: ``Poor man’s Bayes / MAP'' on slide 6).


#### 1.1)

In [4]:
def design_matrix(x):
    """
    Construct a design matrix Φ with a bias (intercept) term.

    Parameters:
    - x: (n,) input array

    Returns:
    - Φ: (n, 2) design matrix with ones in first column and x values in second

    Example:
    >>> x = jnp.array([1., 2., 3.])
    >>> design_matrix(x)
    DeviceArray([[1., 1.],
                 [1., 2.],
                 [1., 3.]], dtype=float32)
    """
    return jnp.column_stack((jnp.ones(len(x)), x, x**2, jnp.sin(0.5 * x)))

In [5]:
x = jnp.array([-2.2, -0.7, 0.0, 1.3, 2.4, 3.0])

y = jnp.array([-0.9, -0.1, 0.2, 1.5, 2.6, 2.7])

In [8]:
Phi = design_matrix(x)
print(f"Design matrix \n{Phi}")

Design matrix 
[[ 1.         -2.2         4.84       -0.8912074 ]
 [ 1.         -0.7         0.48999998 -0.3428978 ]
 [ 1.          0.          0.          0.        ]
 [ 1.          1.3         1.6899998   0.6051864 ]
 [ 1.          2.4         5.76        0.9320391 ]
 [ 1.          3.          9.          0.997495  ]]
